# 🎬 IMDb Sentiment Analysis — Notebook Chạy Thực Nghiệm Trên Google Colab / Kaggle (B1 → D8)

Notebook này thực thi trọn vẹn phần thực nghiệm trên dữ liệu thật cho các task **D1 → D8** (kèm kiểm chứng hợp đồng dữ liệu **B1** và đánh giá test một lần cho **B5**), xuất ra đúng cấu trúc thư mục `experiments/` để `notebooks/presentation.ipynb` và báo cáo đọc trực tiếp.

> **Lưu ý quan trọng trên Colab / Kaggle (theo `docs/phases/TRAINING.md`):**
> - **Không** chạy `pip install -r requirements.txt` (để không ghi đè bản `pandas/pyarrow/torch` có sẵn trên Colab/Kaggle).
> - Chọn Runtime **GPU (T4)** trước khi chạy.

## 1. Khởi tạo Môi trường & Kiểm tra Phần cứng

In [ ]:
import os
import sys
import json
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import pyarrow
import torch
import yaml

# Nếu chạy trên Colab và chưa ở trong repo, tự tìm hoặc hướng dẫn clone
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hw_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Device:", hw_name)
print("pandas:", pd.__version__, "| pyarrow:", pyarrow.__version__, "| sklearn:", sklearn.__version__, "| numpy:", np.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Chuẩn bị & Kiểm chứng Hợp đồng Dữ liệu (B1 — `reviews.parquet`)

Đảm bảo đúng **49.777 dòng** (`train=22.304`, `val=2.473`, `test=25.000`), `0` rò rỉ phim/hash, và `content_hash == 1029824945594009414` theo `experiments/results/data_prep/prepare_stats.json`.

In [ ]:
from src.data.prepare import prepare_dataset, check_contract

parquet_path = PROJECT_ROOT / "data" / "processed" / "reviews.parquet"
if not parquet_path.exists():
    print("⏳ Chưa có reviews.parquet -> tải aclImdb gốc và chạy src.data.prepare...")
    full_df, prep_stats = prepare_dataset()
else:
    full_df = pd.read_parquet(parquet_path)

contract = check_contract(full_df)
content_hash = int(pd.util.hash_pandas_object(full_df, index=False).sum())
expected_hash = json.loads((PROJECT_ROOT / "experiments" / "results" / "data_prep" / "prepare_stats.json").read_text(encoding="utf-8"))["content_hash"]

print("Số dòng theo split:", contract["n_by_split"])
print("content_hash:", content_hash, "| Khớp chuẩn nhóm:", content_hash == expected_hash)
assert contract["n_by_split"] == {"train": 22304, "val": 2473, "test": 25000}
assert content_hash == expected_hash, f"content_hash lệch ({content_hash} != {expected_hash})!"

## 3. Tasks D1, D2, D3 — Vocab, Tham số Mô hình & Ma trận GloVe 100d

In [ ]:
from src.data.tokenizer import tokenize
from src.data.vocab import Vocab
from src.models.rnn import SentimentRNN
from src.models.glove import build_glove_matrix

train_df = full_df[full_df["split"] == "train"].reset_index(drop=True)
vocab_file = PROJECT_ROOT / "data" / "processed" / "vocab.json"
vocab = Vocab.build_from_tokens((tokenize(t) for t in train_df["text_clean"]), min_freq=2, max_size=25000)
vocab.save(vocab_file)
print(f"✅ Đã lưu vocab ({len(vocab):,} từ) tại {vocab_file}")

# Bảng số lượng tham số chính xác cho docs/MODELS.md
specs = [
    ("M1: LSTM", dict(rnn_type="lstm", bidirectional=False, freeze=False, pooling="last")),
    ("M2: GRU", dict(rnn_type="gru", bidirectional=False, freeze=False, pooling="last")),
    ("M3: BiLSTM", dict(rnn_type="lstm", bidirectional=True, freeze=False, pooling="last")),
    ("M4a: BiLSTM + GloVe (frozen)", dict(rnn_type="lstm", bidirectional=True, freeze=True, pooling="last")),
    ("M4b: BiLSTM + GloVe (fine-tune)", dict(rnn_type="lstm", bidirectional=True, freeze=False, pooling="last")),
    ("E1: BiLSTM + Attention", dict(rnn_type="lstm", bidirectional=True, freeze=False, pooling="attention")),
]
param_rows = []
for label, kw in specs:
    m = SentimentRNN(vocab_size=len(vocab), embed_dim=100, hidden_dim=128, **kw)
    c = m.count_parameters()
    param_rows.append({"Model": label, "Total Params": f"{c['total']:,}", "Trainable Params": f"{c['trainable']:,}", "Embedding Trainable": c["embedding_trainable"]})
display(pd.DataFrame(param_rows))

In [ ]:
# Tải GloVe 6B 100d (nếu trên Colab chưa có) và trích xuất data/processed/glove_100d_vocab.npy (Task D3)
import urllib.request
import zipfile

glove_dir = PROJECT_ROOT / "data" / "glove"
glove_dir.mkdir(parents=True, exist_ok=True)
glove_txt = glove_dir / "glove.6B.100d.txt"
glove_npy = PROJECT_ROOT / "data" / "processed" / "glove_100d_vocab.npy"

if not glove_txt.exists():
    zip_path = glove_dir / "glove.6B.zip"
    if not zip_path.exists():
        print("⏳ Đang tải glove.6B.zip từ Stanford NLP...")
        urllib.request.urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip", zip_path)
    print("📦 Đang giải nén glove.6B.100d.txt...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extract("glove.6B.100d.txt", path=glove_dir)

glove_matrix, glove_cov = build_glove_matrix(vocab=vocab, glove_path=glove_txt, embed_dim=100, output_path=glove_npy)
print(f"✅ Đã lưu {glove_npy} | Shape={glove_matrix.shape} | Coverage={glove_cov * 100:.2f}%")

## 4. Task D5 — Pilot M1 (`lstm`, seed=42) kiểm tra mốc Val Acc ≥ 85%

In [ ]:
from src.train import run_training

pilot = run_training(
    config_path=PROJECT_ROOT / "configs" / "lstm.yaml",
    seed=42,
    exp_name_override="pilot_lstm",
)
print(f"📌 Pilot M1 Val Acc = {pilot['best_val_acc']*100:.2f}% tại epoch {pilot['best_epoch']}")
if pilot["best_val_acc"] < 0.85:
    print("⚠️ Pilot < 85%: cân nhắc thử max_len=400 hoặc dropout=0.3 (OVERVIEW.md §2) trước khi chạy D6.")

## 5. Task D6 (+ Mở rộng B E1) — Huấn luyện 6 Cấu hình × 3 Seeds (42, 123, 2025)

In [ ]:
from src.evaluate import evaluate_experiment_seeds, compare_all_models

CONFIG_LIST = [
    "configs/lstm.yaml",                # M1
    "configs/gru.yaml",                 # M2
    "configs/bilstm.yaml",              # M3
    "configs/bilstm_glove_frozen.yaml", # M4a
    "configs/bilstm_glove_ft.yaml",     # M4b
    "configs/bilstm_attention.yaml",    # E1 (Mở rộng B)
]
SEEDS = (42, 123, 2025)

for cfg_rel in CONFIG_LIST:
    cfg_path = PROJECT_ROOT / cfg_rel
    exp_name = cfg_path.stem
    print(f"\n{'='*70}\n🎯 HUẤN LUYỆN {exp_name.upper()} (3 seeds)\n{'='*70}")
    for s in SEEDS:
        run_training(config_path=cfg_path, seed=s)
    # Xuất predictions_val.csv cho từng seed + bản đại diện ở experiments/results/<exp>/predictions_val.csv
    evaluate_experiment_seeds(exp_name=exp_name, split="val", seeds=SEEDS)

val_comp = compare_all_models(split="val")
print("\n🏆 BẢNG SO SÁNH VALIDATION (experiments/results/comparison_val.csv):")
display(val_comp)

In [ ]:
# Chốt mô hình RNN tốt nhất hoàn toàn dựa trên Mean Validation Accuracy (trước khi chạm vào Test)
BEST_RNN_EXP = None
BEST_VAL_MEAN = -1.0
for cfg_rel in CONFIG_LIST:
    exp_name = Path(cfg_rel).stem
    summary_file = PROJECT_ROOT / "experiments" / "results" / exp_name / "summary.json"
    if summary_file.exists():
        s_data = json.loads(summary_file.read_text(encoding="utf-8"))
        m_acc = float(s_data.get("best_val_acc", {}).get("mean", -1.0))
        if m_acc > BEST_VAL_MEAN:
            BEST_VAL_MEAN = m_acc
            BEST_RNN_EXP = exp_name

print(f"⭐ Mô hình RNN chốt theo Val Acc: {BEST_RNN_EXP} (Mean Val Acc = {BEST_VAL_MEAN*100:.2f}%)")

## 6. Task D7 — Mở rộng A: Ablation `max_len` (200 / 400 / 600) & Cách cắt (`post` / `pre` / `head_tail`)

In [ ]:
import copy
from src.evaluate import evaluate_split

base_cfg = yaml.safe_load((PROJECT_ROOT / "configs" / f"{BEST_RNN_EXP}.yaml").read_text(encoding="utf-8"))
variants = [
    ("maxlen_200_post", 200, "post"),
    ("maxlen_400_post", 400, "post"),
    ("maxlen_600_post", 600, "post"),
    ("maxlen_400_pre", 400, "pre"),
    ("maxlen_400_head_tail", 400, "head_tail"),
]
tmp_cfg_dir = PROJECT_ROOT / "experiments" / "runs" / "d7_configs"
tmp_cfg_dir.mkdir(parents=True, exist_ok=True)
train_lens = np.array([len(tokenize(t)) for t in train_df["text_clean"]])

d7_rows = []
for v_name, ml, trunc in variants:
    cfg_v = copy.deepcopy(base_cfg)
    cfg_v["data"]["max_len"] = ml
    cfg_v["data"]["truncation"] = trunc
    cfg_v_path = tmp_cfg_dir / f"{v_name}.yaml"
    cfg_v_path.write_text(yaml.safe_dump(cfg_v, sort_keys=False), encoding="utf-8")

    tr_res = run_training(config_path=cfg_v_path, seed=42, exp_name_override=v_name)
    ev_res = evaluate_split(
        checkpoint_path=tr_res["checkpoint_path"],
        vocab_path=tr_res["vocab_path"],
        split="val",
        output_dir=PROJECT_ROOT / "experiments" / "results" / v_name / "seed_42",
    )
    d7_rows.append({
        "variant": v_name,
        "base_model": BEST_RNN_EXP,
        "max_len": ml,
        "truncation": trunc,
        "pct_train_truncated": round(float((train_lens > ml).mean() * 100), 2),
        "val_accuracy": ev_res["metrics"]["accuracy"],
        "val_macro_f1": ev_res["metrics"]["macro_f1"],
        "val_roc_auc": ev_res["metrics"]["roc_auc"],
        "train_seconds": round(tr_res["total_time"], 1),
        "latency_ms_per_sample": ev_res["speed"]["latency_ms_per_sample"],
    })

d7_df = pd.DataFrame(d7_rows)
d7_csv = PROJECT_ROOT / "experiments" / "results" / "maxlen_comparison.csv"
d7_df.to_csv(d7_csv, index=False)
display(d7_df)

## 7. Task D8 — Đánh giá Tập Test **MỘT LẦN DUY NHẤT**, Bảng So Sánh & Kiểm định McNemar

In [ ]:
from src.models.baseline import evaluate_baseline_on_test
from src.evaluate import run_mcnemar_cli

# 1. Chạy test 1 lần cho Baseline M0a & M0b (với C=10 đã chốt ở B5)
bl_test = evaluate_baseline_on_test()
print("Baseline M0b Test:", bl_test["M0b_tfidf_lr"]["test"]["accuracy"])

# 2. Chạy test 1 lần cho tất cả cấu hình chính (3 seed) + xuất predictions_test.csv bàn giao cho Thư
for cfg_rel in CONFIG_LIST:
    exp_name = Path(cfg_rel).stem
    evaluate_experiment_seeds(exp_name=exp_name, split="test", seeds=SEEDS)

# 3. Xuất comparison_test.csv (tự động gồm M0a, M0b, M1-M4b, E1 và dòng tham chiếu DistilBERT Lab 3)
test_comp = compare_all_models(split="test")
print("\n🏆 BẢNG SO SÁNH TRÊN TẬP TEST (experiments/results/comparison_test.csv):")
display(test_comp)

# 4. Kiểm định McNemar chính xác giữa mô hình RNN tốt nhất (chọn theo Val) và Baseline M0b trên 25.000 mẫu test
best_pred_csv = PROJECT_ROOT / "experiments" / "results" / BEST_RNN_EXP / "predictions_test.csv"
bl_pred_csv = PROJECT_ROOT / "experiments" / "results" / "baseline" / "predictions_test.csv"
mcnemar_res = run_mcnemar_cli(
    pred_file_a=best_pred_csv,
    pred_file_b=bl_pred_csv,
    model_a_name=BEST_RNN_EXP,
    model_b_name="M0b_TFIDF_LR",
)
print("\n📊 KẾT QUẢ MCNEMAR:")
print(json.dumps(mcnemar_res, indent=2, ensure_ascii=False))

## 8. Đóng gói Kết quả Tải về Máy (`experiments/results`, `experiments/plots` & Checkpoint tốt nhất)

In [ ]:
bundle_dir = Path("/tmp/lstm_pj_handover")
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
shutil.copytree(PROJECT_ROOT / "experiments" / "results", bundle_dir / "experiments" / "results")
shutil.copytree(PROJECT_ROOT / "experiments" / "plots", bundle_dir / "experiments" / "plots")

# Kèm checkpoint của mô hình tốt nhất + vocab.json để chạy Demo Gradio (B6) và Explain (T3) trên CPU
best_ckpt_src = PROJECT_ROOT / "experiments" / "checkpoints" / BEST_RNN_EXP
if best_ckpt_src.exists():
    shutil.copytree(best_ckpt_src, bundle_dir / "experiments" / "checkpoints" / BEST_RNN_EXP)

zip_base = PROJECT_ROOT.parent / "LSTM_PJ_colab_handover"
zip_file = shutil.make_archive(str(zip_base), "zip", root_dir=bundle_dir)
print(f"✅ Đã đóng gói toàn bộ kết quả bàn giao tại: {zip_file} ({Path(zip_file).stat().st_size / 1e6:.1f} MB)")